# Phase 5 — Showcase Evaluation

Generalization test: apply the trained meta-learner to 9 held-out showcase datasets
that were never seen during meta-training.

**Procedure per showcase dataset**:
1. Extract Option A meta-features
2. Predict best clustering method (meta-classifier)
3. Predict expected LSE per method (meta-regressor)
4. Run the **predicted** method, the **oracle** (all 6, pick best), and **k-means** baseline
5. Compute true LSE for each and build comparison table

**Output**: `outputs/figures/showcase_comparison.csv` + printed comparison table

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR     = os.path.join(ROOT, 'data', 'raw')
MODELS_DIR  = os.path.join(ROOT, 'outputs', 'models')
FIGURES_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(ROOT, 'src'))

openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)

print('Paths OK')

In [ ]:
from lse import compute_lse, groundtruth_accuracy
from clustering import (
    pseudo_kmeans, pseudo_dbscan, pseudo_agglomerative,
    pseudo_gmm, pseudo_autoencoder, pseudo_dictlearn,
)
from metafeatures import extract_optA

print('Imports OK')

In [ ]:
# ── Showcase dataset IDs (NEVER in meta-training) ─────────────────────────────
# OpenML IDs for the 9 showcase datasets defined in CLAUDE.md
SHOWCASE_DATASETS = [
    {'id': 61,    'name': 'Iris'},
    {'id': 187,   'name': 'Wine'},
    {'id': 15,    'name': 'Breast Cancer Wisconsin'},
    {'id': 53,    'name': 'Heart Disease (UCI)'},
    {'id': 40966, 'name': 'Palmer Penguins'},
    {'id': 37,    'name': 'Diabetes (Pima)'},
    {'id': 54,    'name': 'Vehicle Silhouettes'},
    {'id': 1590,  'name': 'Adult Income'},
    {'id': 1597,  'name': 'Credit Card Fraud'},
]

SHOWCASE_IDS = {d['id'] for d in SHOWCASE_DATASETS}

METHODS = {
    'kmeans'   : pseudo_kmeans,
    'dbscan'   : pseudo_dbscan,
    'agg'      : pseudo_agglomerative,
    'gmm'      : pseudo_gmm,
    'autoenc'  : pseudo_autoencoder,
    'dictlearn': pseudo_dictlearn,
}

METHOD_NAMES = list(METHODS.keys())
print(f'{len(SHOWCASE_DATASETS)} showcase datasets, {len(METHODS)} methods')

In [ ]:
# ── Load trained meta-learner models ─────────────────────────────────────────
clf_bundle = joblib.load(os.path.join(MODELS_DIR, 'meta_clf_optA.pkl'))
reg_bundle = joblib.load(os.path.join(MODELS_DIR, 'meta_reg_optA.pkl'))

meta_clf      = clf_bundle['pipeline']
clf_feat_cols = clf_bundle['feature_cols']

meta_reg      = reg_bundle['pipeline']
reg_feat_cols = reg_bundle['feature_cols']
lse_cols      = reg_bundle['lse_cols']       # e.g. ['LSE_kmeans', 'LSE_dbscan', ...]

print('Classifier features:', clf_feat_cols)
print('Regressor  features:', reg_feat_cols)
print('LSE cols in reg    :', lse_cols)

In [ ]:
# ── Helper: load + split + scale ─────────────────────────────────────────────

def load_and_split(dataset_id):
    ds = openml.datasets.get_dataset(
        dataset_id,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [ ]:
# ── Main evaluation loop ──────────────────────────────────────────────────────

records = []

for ds_info in SHOWCASE_DATASETS:
    did  = ds_info['id']
    name = ds_info['name']
    print(f'\n── {name} (id={did}) ──')

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        if X_tr.shape[1] == 0:
            print('  SKIP: no numeric features')
            continue
        X_tr_sc, X_te_sc = scale(X_tr, X_te)
        n_cls = len(np.unique(y_tr))

        # Ground-truth RF accuracy (denominator for all LSE)
        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)
        print(f'  gt_acc={gt_acc:.3f}  n_cls={n_cls}  n_tr={len(X_tr)}')

        # ── Step 1: Extract meta-features ─────────────────────────────────────
        # Pass raw unscaled data — extract_optA applies StandardScaler internally
        feats = extract_optA(X_tr, y_tr, X_te, y_te)
        feat_vec_clf = np.array([[feats.get(c, np.nan) for c in clf_feat_cols]])
        feat_vec_reg = np.array([[feats.get(c, np.nan) for c in reg_feat_cols]])

        # ── Step 2: Predict best method (classifier) ──────────────────────────
        predicted_method = meta_clf.predict(feat_vec_clf)[0]
        print(f'  Predicted best method: {predicted_method}')

        # ── Step 3: Predict LSE per method (regressor) ────────────────────────
        predicted_lse_vec = meta_reg.predict(feat_vec_reg)[0]   # shape (n_lse_cols,)
        predicted_lse = dict(zip(
            [c.replace('LSE_', '') for c in lse_cols],
            predicted_lse_vec,
        ))
        print(f'  Predicted LSE:  ' +
              '  '.join(f'{m}={v:.3f}' for m, v in predicted_lse.items()))

        # ── Step 4: Run all 6 methods → true LSE ─────────────────────────────
        true_lse = {}
        for mname, fn in METHODS.items():
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse_val, _ = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=mname, dataset_name=name, verbose=False,
                )
                true_lse[mname] = round(lse_val, 4)
            except Exception as e:
                print(f'    {mname} FAILED: {e}')
                true_lse[mname] = np.nan

        # ── Step 5: Compute strategy outcomes ────────────────────────────────
        oracle_method = max(
            (m for m in true_lse if not np.isnan(true_lse[m])),
            key=lambda m: true_lse[m],
        )
        oracle_lse    = true_lse[oracle_method]
        predicted_lse_actual = true_lse.get(predicted_method, np.nan)
        kmeans_lse    = true_lse.get('kmeans', np.nan)

        print(f'  Oracle  : {oracle_method} → LSE={oracle_lse:.3f}')
        print(f'  Predicted ({predicted_method}): LSE={predicted_lse_actual:.3f}')
        print(f'  k-means baseline: LSE={kmeans_lse:.3f}')

        rec = {
            'dataset'               : name,
            'dataset_id'            : did,
            'n_classes'             : n_cls,
            'gt_acc'                : round(gt_acc, 3),
            'predicted_method'      : predicted_method,
            'predicted_lse_actual'  : round(predicted_lse_actual, 3) if not np.isnan(predicted_lse_actual) else np.nan,
            'oracle_method'         : oracle_method,
            'oracle_lse'            : round(oracle_lse, 3),
            'kmeans_lse'            : round(kmeans_lse, 3) if not np.isnan(kmeans_lse) else np.nan,
            'hit'                   : predicted_method == oracle_method,
            'gap_vs_oracle'         : round(oracle_lse - predicted_lse_actual, 3) if not np.isnan(predicted_lse_actual) else np.nan,
            'gain_vs_kmeans'        : round(predicted_lse_actual - kmeans_lse, 3) if not np.isnan(predicted_lse_actual) else np.nan,
        }
        # Attach all true LSE values for reference
        for m, v in true_lse.items():
            rec[f'true_lse_{m}'] = v

        records.append(rec)

    except Exception as e:
        print(f'  FAILED: {e}')

print('\nDone.')

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(records)

SUMMARY_COLS = [
    'dataset', 'n_classes', 'gt_acc',
    'predicted_method', 'predicted_lse_actual',
    'oracle_method', 'oracle_lse',
    'kmeans_lse',
    'hit', 'gap_vs_oracle', 'gain_vs_kmeans',
]

summary = results_df[SUMMARY_COLS].copy()
print(summary.to_string(index=False))

print(f'\nTop-1 accuracy  : {summary["hit"].mean():.1%}  ({summary["hit"].sum()}/{len(summary)})')
print(f'Mean gap vs oracle   : {summary["gap_vs_oracle"].mean():.3f}')
print(f'Mean gain vs k-means : {summary["gain_vs_kmeans"].mean():.3f}')

In [ ]:
# ── Save to CSV ───────────────────────────────────────────────────────────────
OUT_PATH = os.path.join(FIGURES_DIR, 'showcase_comparison.csv')
results_df.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')